# Device Simulator — REST API

Full end-to-end simulation of one HVAC device pushing sensor data via the QoE REST API.

**Flow:**
1. User logs in → user JWT (needed for admin operations)
2. List available devices
3. Upsert device credentials (get `device_key` + `device_secret`)
4. Device authenticates → device JWT
5. Inspect device sensors
6. Push a single sensor reading
7. Push a bulk batch
8. Continuous stream simulation

## 1. Import Required Libraries

In [1]:
import requests
import json
import uuid
import time
import random
from datetime import datetime, timezone
from pprint import pprint

SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json"})
print("Libraries loaded.")

Libraries loaded.


## 2. Configure REST API Connection

### ⚡ Quickest option — copy from the UI

1. Open the **QoE UI** → *Settings → Installed Devices*
2. Click **View Details** on your target device
3. Scroll to the **Simulator Config** section → **REST Notebook** tab
4. Click **Rotate & Reveal Secret** to generate / expose the `device_secret`
5. Click **Copy** and **paste the block directly into the cell below** (replace the whole cell body)

The pasted snippet already contains `API_BASE`, `DEVICE_ID`, `DEVICE_KEY`, `DEVICE_SECRET`, `BUILDING_ID`, and the `SENSORS` list — no further editing needed.  
When `SENSORS` is pre-populated, step **7 (Inspect Sensors)** is skipped automatically.

---

### Manual option

Fill in `DEVICE_KEY` and `DEVICE_SECRET` from a previous key rotation, set `DEVICE_ID` to the integer device id visible in the UI, and leave `SENSORS = []` so the notebook fetches the list for you.

In [17]:
# ── Paste the "REST Notebook" snippet from UI → Device Details → Simulator Config ──
# ── OR fill in the values below manually ────────────────────────────────────────────
API_BASE      = "http://localhost:8000"
DEVICE_ID     = 1
DEVICE_KEY    = "6149c806-4cae-4ddc-b869-67d3f0976f57"
DEVICE_SECRET = "VTgHih6Pq-61iuPhM4nhR-HoKROU_kZQGVLt5zqHh6qXfu-hxMywfQwDpQIx1vsN"       # rotate key in UI → Device Details → Technical Documentation
BUILDING_ID   = 1
SENSORS       = [
    {"id": 1, "name": "temperature-1", "sensor_type": "temperature", "unit": "°C",     "building_id": 1},
    {"id": 2, "name": "energy-1",      "sensor_type": "energy",      "unit": "kWh",    "building_id": 1},
    {"id": 3, "name": "presence-1",    "sensor_type": "presence",    "unit": "binary", "building_id": 1},
]

# User JWT — paste here if you already have one, otherwise run the login cell below
USER_JWT = ""
# ────────────────────────────────────────────────────────────────────────────────────
print(f"API base : {API_BASE}")
print(f"Device ID: {DEVICE_ID}  |  Building ID: {BUILDING_ID}")
print(f"Device Key: {DEVICE_KEY}")
if SENSORS:
    print(f"Sensors  : {len(SENSORS)} pre-loaded from config")
    for s in SENSORS:
        print(f"  id={s['id']}  type={s['sensor_type']:15s}  unit={s['unit']:6s}  # {s.get('name','')}")

API base : http://localhost:8000
Device ID: 1  |  Building ID: 1
Device Key: 6149c806-4cae-4ddc-b869-67d3f0976f57
Sensors  : 3 pre-loaded from config
  id=1  type=temperature      unit=°C      # temperature-1
  id=2  type=energy           unit=kWh     # energy-1
  id=3  type=presence         unit=binary  # presence-1


## 3. User Login (skip if you already have a JWT)

Use traditional login with one of the mock users (`Tester1` … `Tester4`).  
> **Web3 mode**: paste your JWT from the browser into `USER_JWT` in the config cell above.

In [3]:
if not USER_JWT:
    # Traditional login — works when AUTH_TYPE=jwt and mock data is loaded
    LOGIN_USERNAME = "Tester1"   # mock user created by mock_data.py
    LOGIN_PASSWORD = "changeme"  # update if your mock data sets a different password

    r = SESSION.post(f"{API_BASE}/auth/login", json={
        "username": LOGIN_USERNAME,
        "password": LOGIN_PASSWORD,
    })
    if r.status_code == 200:
        body = r.json()
        USER_JWT = body.get("token") or body.get("access_token", "")
        print(f"✅ Logged in — token: {USER_JWT[:40]}...")
    else:
        print(f"❌ Login failed ({r.status_code}): {r.text}")
        print("   → Paste your JWT into USER_JWT in the config cell above.")
else:
    print("✅ Using pre-configured USER_JWT.")

❌ Login failed (401): {"detail":"External authentication failed"}
   → Paste your JWT into USER_JWT in the config cell above.


## 4. List Devices & Pick One

In [18]:
auth_header = {"Authorization": f"Bearer {USER_JWT}"}

r = SESSION.get(f"{API_BASE}/devices/", headers=auth_header)
assert r.status_code == 200, f"GET /devices/ failed: {r.status_code} {r.text}"

devices = r.json()
print(f"Found {len(devices)} device(s):\n")
for d in devices:
    print(f"  id={d['id']}  name={d['name']}  building={d['building_name']}  "
          f"sensors={d['sensor_count']}  key={d['device_key']}")

# Auto-pick if not set
if DEVICE_ID is None:
    DEVICE_ID = devices[0]["id"]
    print(f"\nAuto-selected device id={DEVICE_ID}")

AssertionError: GET /devices/ failed: 401 {"detail":"Not authenticated"}

## 5. Get / Upsert Device Credentials

Generates a fresh `device_key` + `device_secret` for the selected device.  
Skip this cell if you already have credentials in the config.

In [5]:
if not DEVICE_KEY or not DEVICE_SECRET:
    r = SESSION.post(
        f"{API_BASE}/devices/{DEVICE_ID}/credentials/upsert",
        headers=auth_header,
        json={},
    )
    assert r.status_code == 200, f"Credential upsert failed: {r.status_code} {r.text}"
    creds = r.json()
    DEVICE_KEY    = creds["device_key"]
    DEVICE_SECRET = creds["device_secret"]
    print(f"✅ New credentials generated")
else:
    print("✅ Using pre-configured credentials")

print(f"   device_key    = {DEVICE_KEY}")
print(f"   device_secret = {DEVICE_SECRET[:10]}...")

✅ Using pre-configured credentials
   device_key    = 6149c806-4cae-4ddc-b869-67d3f0976f57
   device_secret = VTgHih6Pq-...


## 6. Device Authentication → Device JWT

The device uses its own credentials (not the user JWT) to get a short-lived device JWT (`POST /auth/device/auth`).

In [19]:
r = SESSION.post(f"{API_BASE}/auth/device/auth", json={
    "device_key":    DEVICE_KEY,
    "device_secret": DEVICE_SECRET,
})
assert r.status_code == 200, f"Device auth failed: {r.status_code} {r.text}"

DEVICE_JWT = r.json()["access_token"]
DEVICE_HEADER = {"Authorization": f"Bearer {DEVICE_JWT}"}
print(f"✅ Device JWT obtained: {DEVICE_JWT[:50]}...")

✅ Device JWT obtained: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJzdWIiOiIxI...


## 7. Inspect Device Sensors

Get the list of sensors registered to this device so we know which `sensor_id` values to use.

In [7]:
if SENSORS:
    # Already provided by the UI snippet — skip the API call
    print(f"✅ Using {len(SENSORS)} sensor(s) from config:")
    for s in SENSORS:
        print(f"  id={s['id']:3d}  type={s.get('sensor_type','?'):15s}  building_id={s['building_id']}")
    SENSOR_MAP = {s["sensor_type"]: s["id"] for s in SENSORS}
    if BUILDING_ID is None:
        BUILDING_ID = SENSORS[0]["building_id"]
else:
    r = SESSION.get(f"{API_BASE}/devices/{DEVICE_ID}/sensors", headers=auth_header)
    assert r.status_code == 200, f"GET sensors failed: {r.status_code} {r.text}"

    SENSORS = r.json()
    print(f"Device {DEVICE_ID} has {len(SENSORS)} sensor(s):\n")
    for s in SENSORS:
        print(f"  id={s['id']:3d}  type={s.get('sensor_type','?'):15s}  unit={s.get('unit','?'):10s}  name={s['name']}")

    SENSOR_MAP = {s["sensor_type"]: s["id"] for s in SENSORS}
    BUILDING_ID = SENSORS[0].get("building_id") if SENSORS else BUILDING_ID
    print(f"\nBuilding id: {BUILDING_ID}")

✅ Using 3 sensor(s) from config:
  id=  1  type=temperature      building_id=1
  id=  2  type=energy           building_id=1
  id=  3  type=presence         building_id=1


## 8. Define Device Payload Structure & Helper

Build realistic sensor readings that match `SensorDataCreate`.

In [8]:
# Maps sensor_type to a value generator
_VALUE_GEN = {
    "temperature": lambda: round(random.uniform(18.0, 28.0), 2),
    "energy":      lambda: round(random.uniform(0.5, 5.0), 3),
    "presence":    lambda: None,   # bool sensor
    "humidity":    lambda: round(random.uniform(30.0, 70.0), 1),
    "power":       lambda: round(random.uniform(100, 3000), 1),
    "current":     lambda: round(random.uniform(0.5, 15.0), 2),
    "voltage":     lambda: round(random.uniform(220.0, 240.0), 1),
}

def make_reading(sensor: dict, ts: datetime | None = None) -> dict:
    """Build a SensorDataCreate-compatible dict for one sensor."""
    ts = ts or datetime.now(timezone.utc)
    stype = sensor.get("sensor_type", "")
    gen = _VALUE_GEN.get(stype, lambda: round(random.uniform(0, 100), 2))
    raw = gen()
    payload = {
        "sensor_id":  sensor["id"],
        "building_id": sensor["building_id"],
        "ts":          ts.isoformat(),
        "quality":     "valid",
    }
    if stype == "presence":
        payload["value_bool"] = random.choice([True, False])
    elif isinstance(raw, str):
        payload["value_text"] = raw
    else:
        payload["value"] = raw
    return payload

# Quick sanity check
sample = make_reading(SENSORS[0])
print("Sample reading:")
pprint(sample)

Sample reading:
{'building_id': 1,
 'quality': 'valid',
 'sensor_id': 1,
 'ts': '2026-05-13T14:22:13.452139+00:00',
 'value': 18.13}


## 9. Send a Single Sensor Reading

`POST /sensordata/` — authenticated with the **device JWT**.

In [9]:
reading = make_reading(SENSORS[0])
print("Sending:")
pprint(reading)

r = SESSION.post(
    f"{API_BASE}/sensordata/",
    headers={**DEVICE_HEADER, "Content-Type": "application/json"},
    json=reading,
)
print(f"\nStatus: {r.status_code}")
if r.status_code == 200:
    print("✅ Accepted")
    pprint(r.json())
else:
    print(f"❌ {r.text}")

Sending:
{'building_id': 1,
 'quality': 'valid',
 'sensor_id': 1,
 'ts': '2026-05-13T14:22:16.355159+00:00',
 'value': 26.64}

Status: 200
✅ Accepted
{'building_id': 1,
 'id': 118,
 'quality': 'valid',
 'sensor_id': 1,
 'ts': '2026-05-13T14:22:16.355159Z',
 'value': 26.64,
 'value_bool': None,
 'value_text': None}


## 10. Send Bulk Sensor Readings

`POST /sensordata/bulk` — all sensors of the device in one request (up to 1000 records).

In [10]:
ts = datetime.now(timezone.utc)
batch = [make_reading(s, ts) for s in SENSORS]
print(f"Sending bulk batch of {len(batch)} readings…")

r = SESSION.post(
    f"{API_BASE}/sensordata/bulk",
    headers={**DEVICE_HEADER, "Content-Type": "application/json"},
    json=batch,
)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    print(f"✅ Accepted {len(r.json())} rows")
else:
    print(f"❌ {r.text}")

Sending bulk batch of 3 readings…
Status: 200
✅ Accepted 3 rows


## 11. Handle API Response — Inspect Stored Data

Read back the last N sensor readings for verification.

In [11]:
# GET /sensordata/me — device JWT only, no user login needed
SENSOR_ID = SENSORS[0]["id"]
r = SESSION.get(
    f"{API_BASE}/sensordata/me",
    headers=DEVICE_HEADER,
    params={"sensor_id": SENSOR_ID, "limit": 5},
)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    rows = r.json()
    if rows:
        print(f"Last {len(rows)} aggregated reading(s) for sensor {SENSOR_ID}:")
        for row in rows:
            print(f"  ts={row.get('ts')}  value={row.get('value')}  "
                  f"value_bool={row.get('value_bool')}  quality={row.get('quality')}")
    else:
        print("⚠️  No rows yet — pg_cron aggregates raw data every minute. Wait ~1 min and re-run.")
else:
    print(f"❌ {r.text}")

Status: 200
Last 5 aggregated reading(s) for sensor 1:
  ts=2026-05-15T23:55:00Z  value=22.65  value_bool=None  quality=valid
  ts=2026-05-15T23:50:00Z  value=22.6  value_bool=None  quality=valid
  ts=2026-05-15T23:45:00Z  value=22.53  value_bool=None  quality=valid
  ts=2026-05-15T23:40:00Z  value=22.47  value_bool=None  quality=valid
  ts=2026-05-15T23:35:00Z  value=22.41  value_bool=None  quality=valid


## 12. Continuous Stream Simulation

Sends one bulk batch every `INTERVAL_SEC` seconds for `ROUNDS` rounds.  
`Ctrl+C` / interrupt kernel to stop early.

In [ ]:
INTERVAL_SEC = 5    # seconds between each push
ROUNDS       = 12   # total pushes (set to 0 for infinite)

def _reauth_if_needed():
    """Re-authenticate the device if the JWT is close to expiry (every 25 min)."""
    global DEVICE_JWT, DEVICE_HEADER, _last_auth
    if time.time() - _last_auth > 25 * 60:
        r = SESSION.post(f"{API_BASE}/auth/device/auth", json={
            "device_key": DEVICE_KEY, "device_secret": DEVICE_SECRET
        })
        if r.status_code == 200:
            DEVICE_JWT = r.json()["access_token"]
            DEVICE_HEADER = {"Authorization": f"Bearer {DEVICE_JWT}"}
            _last_auth = time.time()
            print("   🔄 Device token refreshed")

_last_auth = time.time()
count = 0

try:
    while ROUNDS == 0 or count < ROUNDS:
        _reauth_if_needed()
        ts  = datetime.now(timezone.utc)
        batch = [make_reading(s, ts) for s in SENSORS]
        r = SESSION.post(
            f"{API_BASE}/sensordata/bulk",
            headers={**DEVICE_HEADER, "Content-Type": "application/json"},
            json=batch,
        )
        status = "✅" if r.status_code == 200 else f"❌ {r.status_code}"
        print(f"[{ts.strftime('%H:%M:%S')}] round={count+1:4d}  sensors={len(batch)}  {status}")
        count += 1
        if ROUNDS == 0 or count < ROUNDS:
            time.sleep(INTERVAL_SEC)
except KeyboardInterrupt:
    print("\n⏹ Stream stopped.")

print(f"\nDone — sent {count} round(s).")

In [16]:
from datetime import timedelta

TS_FROM = (datetime.now(timezone.utc) - timedelta(hours=1)).isoformat()  # last 1 hour
TS_TO   = datetime.now(timezone.utc).isoformat()                          # now

r = SESSION.get(
    f"{API_BASE}/sensordata/me",
    headers=DEVICE_HEADER,
    params={"limit": 1500, "ts_from": TS_FROM, "ts_to": TS_TO},
)
print(f"Status: {r.status_code}")
print(f"Range : {TS_FROM}  →  {TS_TO}\n")
if r.status_code == 200:
    rows = r.json()
    if rows:
        print(f"{'sensor_id':>10}  {'ts':<30}  {'value':>10}  {'value_bool':>10}  quality")
        print("-" * 80)
        for row in rows:
            print(f"{row.get('sensor_id'):>10}  {str(row.get('ts')):<30}  "
                  f"{str(row.get('value') or ''):>10}  {str(row.get('value_bool') or ''):>10}  {row.get('quality')}")
    else:
        print("⚠️  No rows in this range — wait ~1 min for pg_cron and re-run.")
else:
    print(f"❌ {r.text}")

Status: 401
Range : 2026-05-13T14:08:14.442359+00:00  →  2026-05-13T15:08:14.442359+00:00

❌ {"detail":"Invalid or expired device token"}
